# EDA - Sentimiento de YouTube vs Zonas Turísticas
Analizar los comentarios negativos y extraer tópicos frecuentes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

In [ ]:
# 1. Conectar a PostgreSQL (esquema silver) para obtener los resultados del análisis de sentimiento
# Asegúrate de tener cargado el archivo .env
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv('../.env')
engine = create_engine(f"postgresql://{os.getenv('AZURE_DB_USER')}:{os.getenv('AZURE_DB_PASSWORD')}@{os.getenv('AZURE_DB_HOST')}:5432/{os.getenv('AZURE_DB_NAME')}?sslmode=require")

# Si aún no tienes la tabla en Azure, puedes leer el parquet local de reseñas si lo tienes.
try:
    df = pd.read_sql('SELECT * FROM silver.sentiment_results', engine)
    display(df.head())
except Exception as e:
    print("La tabla silver.sentiment_results aún no existe en Azure. ¡Súbela primero con dbt!")


In [ ]:
# 2. Distribución de sentimientos (ejemplo con datos simulados si falla la DB)
if 'df' in locals():
    sns.countplot(data=df, x='label', palette='viridis')
    plt.title('Distribución de Sentimiento en YouTube')
    plt.show()

In [ ]:
# 3. Extraer palabras más comunes en comentarios NEGATIVOS
if 'df' in locals():
    negativos = df[df['label'] == 'negative']['text'].dropna()

    palabras = []
    for text in negativos:
        # Limpiar signos de puntuación y pasar a minúsculas
        limpio = re.sub(r'[^\w\s]', '', text.lower())
        palabras.extend(limpio.split())

    # Filtrar palabras cortas (stopwords básicas)
    palabras_filtradas = [p for p in palabras if len(p) > 4]

    count = Counter(palabras_filtradas)
    print('Top 20 palabras en quejas/comentarios negativos:')
    for word, freq in count.most_common(20):
        print(f'{word}: {freq}')